#Data Audit - Retail Orders Dataset


This notebook checks the raw dataset before cleaning.


Main checks:
1. Dataset shapes
2. Column names
3. Data types
4. Missing values
5. Duplicate records
6. Date format issues
7. Numeric issues
8. Text/category issues
9. Total price validation

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
raw_data_path = Path("../data/raw/Dataset for Data Analytics.xlsx") #define file path
df = pd.read_excel(raw_data_path) #load excel file
df.head() #display first 5 rows

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [8]:
df.shape #check number of rows and columns
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 1200
Number of columns: 14


In [9]:
df.columns #display all columns names
for column in df.columns: #show column names in a clean list
    print(column)

OrderID
Date
CustomerID
Product
Quantity
UnitPrice
ShippingAddress
PaymentMethod
OrderStatus
TrackingNumber
ItemsInCart
CouponCode
ReferralSource
TotalPrice


In [10]:
df.dtypes #check data types of each column

OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

In [12]:
missing_values = df.isnull().sum() #count missing values in each column
missing_values

OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64

In [15]:
missing_percentage = (df.isnull().sum() / len(df))*100
missing_report = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage.round(2)
})
missing_report.sort_values(by="Missing Count", ascending=False)

,Missing Count,Missing Percentage
CouponCode,309,25.75
OrderID,0,0.00
Date,0,0.00
CustomerID,0,0.00
Product,0,0.00
Quantity,0,0.00
UnitPrice,0,0.00
ShippingAddress,0,0.00
PaymentMethod,0,0.00
OrderStatus,0,0.00


In [16]:
duplicate_rows = df.duplicated().sum() #check duplicate rows
print(f"Number of Duplicate rows: {duplicate_rows}")

Number of Duplicate rows: 0


In [17]:
duplicate_order_ids = df["OrderID"].duplicated().sum()
print(f"Number of duplicate Order ID values: {duplicate_order_ids}")

Number of duplicate Order ID values: 0


In [18]:
df["Date"].head(10) #display first 10 date values

0   2023-01-04
1   2024-08-23
2   2024-02-27
3   2023-10-15
4   2025-05-08
5   2023-10-23
6   2025-06-17
7   2023-05-12
8   2025-04-02
9   2023-11-21
Name: Date, dtype: datetime64[ns]

In [13]:
# Check whether Date column has invalid dates
converted_dates = pd.to_datetime(df["Date"], errors="coerce")

invalid_dates = converted_dates.isnull().sum()

print(f"Number of invalid dates: {invalid_dates}")

Number of invalid dates: 0


In [4]:
df.describe() #check basic statistics for numeric columns

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [5]:
expected_total = df["Quantity"] * df["UnitPrice"] 
price_difference = df["TotalPrice"] - expected_total
incorrect_total_price = (price_difference.round(2) != 0).sum() #count rows where total price is wrong
print(f"Rows with incorrect TotalPrice: {incorrect_total_price}")

Rows with incorrect TotalPrice: 0


In [6]:
text_columns = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource"]
for col in text_columns:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))


Column: Product
Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

Column: PaymentMethod
PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

Column: OrderStatus
OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

Column: ReferralSource
ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64


In [14]:
audit_summary = {
    "Total Rows": df.shape[0],
    "Total Columns": df.shape[1],
    "Duplicate Rows": df.duplicated().sum(),
    "Duplicate OrderID": df["OrderID"].duplicated().sum(),
    "Missing Values Total": df.isnull().sum().sum(),
    "Invalid Dates": invalid_dates,
    "Incorrect TotalPrice Rows": incorrect_total_price
}

audit_summary


{'Total Rows': 1200,
 'Total Columns': 14,
 'Duplicate Rows': np.int64(0),
 'Duplicate OrderID': np.int64(0),
 'Missing Values Total': np.int64(309),
 'Invalid Dates': np.int64(0),
 'Incorrect TotalPrice Rows': np.int64(0)}

In [15]:
audit_summary_df = pd.DataFrame(list(audit_summary.items()), columns = ["Check", "Result"])
audit_summary_df

,Check,Result
0,Total Rows,1200
1,Total Columns,14
2,Duplicate Rows,0
3,Duplicate OrderID,0
4,Missing Values Total,309
5,Invalid Dates,0
6,Incorrect TotalPrice Rows,0
